In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter1d

In [ ]:
# suggested values
alpha_val = 0.9
beta_val = 0.2
small_gamma_val = 0.1
lambda_0 = 1
lambda_1 = 5

n = 10
big_T = 100

Plan:
* make transition matrix
* make emission matrix
* make pi?
* simulate (research how)

In [ ]:

# C_t = 0: serial stimuli 0
# C_t = 1: serial stimuli 1
# C_t = 2: parallel both stimuli 0 & 1

def create_transition_matrix(gamma: float, beta: float):
    Gamma = np.array([[1 - gamma,  0,          gamma], 
                      [0,          1 - gamma,  gamma], 
                      [beta/2,     beta/2,     1 - beta]])
    return Gamma

## Emission - conditional $Z_{t,i}$
Since our model has multiple layers, we don't directly have an emission matrix from $C_t$ to $X_{t,i}$. Instead we need to go through $Z_{t,i}$ up to $C_t$, which is given as the conditional:
$$
P(Z_{t,i} = 1 | C_t = c) = \begin{cases}
1 - \alpha & \text{if } c = 0 \\
\alpha & \text{if } c = 1 \\
0.5 & \text{if } c = 2
\end{cases}
$$


In [ ]:
def p_z_given_c(alpha, c: int): # z_{t,i} = 1 | C_t = c
    if c == 0:
        return 1 - alpha
    elif c == 1:
        return alpha
    elif c == 2:
        return 0.5
    else:
        raise ValueError("Error with input, only c = 0, c = 1 or c = 2 accepted.")

In [ ]:
# p_z_given_c(alpha=alpha_val, c=2)

In [ ]:
test_matrix = create_transition_matrix(gamma=small_gamma_val, beta=beta_val)
print(test_matrix)

## Random Walk for $n = 10$, and $T = 100$

In [ ]:

# * suggested values
# alpha_val = 0.9
# beta_val = 0.2
# small_gamma_val = 0.1
# lambda_0 = 1
# lambda_1 = 5

# n = 10
# big_T = 100

def simulate_c(gamma, beta, T):
    Gamma = create_transition_matrix(gamma=gamma, beta=beta)
    C = np.empty(T, dtype=int) # empty matrix to hold simulated values
    C[0] = 2 # first entry is always 2
    
    c_states = np.array([0,1,2], dtype=int) # possible states C_t can take on
    
    for t in range(1, T):
        # depending on the value the previous C had, we do a choice for where to go next given the distribution
        # so if we start at 2, our C[T-1] = 2, and so we'd access p=Gamma[2], and sample a random choice of those (giving us [0,1,2] as possible outcomes)
        # and select that value as the next C[T].
        C[t] = np.random.choice(a=c_states, p=Gamma[C[t-1]]) 
        
    return C
        

In [ ]:
# C = simulate_c(gamma=small_gamma_val, beta=beta_val, T=big_T)
# C

In [ ]:
def simulate_z(C, alpha, n):
    T = len(C) # to ensure we don't give a new T which would break our previous (shapes)
    Z = np.empty(shape=(T, n), dtype=int) # before we only had one C per time slice, but now we have a Z per neuron, of which there are n, so we need a matrix instead of an array
    for t in range(0, T):
        z_prob = p_z_given_c(alpha=alpha, c=C[t]) # all Z's have the same C (share parent), so we only get a single C per row in matri x
        Z[t] = np.random.binomial(n=1, p=z_prob, size=n) # and now we do a random draw per z, that is for each row we do n random bernoulli samples (z \in \brc{0, 1})
    return Z

In [ ]:
def simulate_x(Z, lam0, lam1):
    T = len(Z)
    n = len(Z[0])
    X = np.empty(shape=(T,n), dtype=int)
    for t in range(0, T):
        lambdas = np.where(Z[t] == 0, lam0, lam1)
        X[t] = np.random.poisson(lam=lambdas, size=n)
    return X

### Simulation run

In [ ]:
C = simulate_c(gamma=small_gamma_val, beta=beta_val, T=big_T)
Z = simulate_z(C=C, alpha=alpha_val, n=n)
X = simulate_x(Z=Z, lam0=lambda_0, lam1=lambda_1)

print(C.shape)
print(Z.shape)
print(X)

In [ ]:
print(X[:,0])

In [ ]:
X.shape[1]

In [ ]:
def plotter(X, show_mean: bool):
    n = X.shape[1]
    t_vals = np.arange(1, X.shape[0]+1)
    fig, axes = plt.subplots(n, 1, figsize=(12, 1.5*n))
    fig.suptitle("Plots of generated data", fontsize=15)
    
    x_ticks = np.r_[1, np.arange(10, 101, 10)] # makes [1, array([10, 20, ...])] where r_ turns it into [1, 10, 20, ...]
    cmap = ListedColormap([
        'deepskyblue','blue','navy','blueviolet','indigo',
        'purple','mediumvioletred','magenta','crimson','r'
    ])
    colors = cmap.colors
    for i in range(0,n):
        axes[i].plot(t_vals, X[:,i], marker="o", markersize=4, color=colors[i % len(colors)], zorder=2)
        axes[i].set_xlabel("t")
        axes[i].set_xlim(1,100)
        axes[i].set_xticks(x_ticks)
        y_top = max(10, X[:, i].max()) # so we ensure the y ticks has 0, 5, 10 and not just 0, 5
        axes[i].set_ylim(0,y_top+1) # range of y ticks. If we want to have the same range for all plots, change to static values or the highest value in entire dataset
        axes[i].set_ylabel(f"$X_{{t,{i+1}}}$")
        axes[i].yaxis.set_minor_locator(MultipleLocator(5))
        axes[i].grid(which='both',alpha=0.3, zorder=1)
        
        if show_mean: # * shows mean curve across all subplots
            mean_X = X.mean(axis=1)
            smooth_mean = gaussian_filter1d(mean_X, sigma=5)
            axes[i].plot(t_vals, smooth_mean, color='darkgray', zorder=1)

    plt.tight_layout()
    plt.show()

plotter(X, show_mean=False)

In [ ]:
def mean_plotter(X):
    plt.figure(figsize=(12,3))
    t_vals = np.arange(1, X.shape[0]+1)
    mean_X = X.mean(axis=1)
    smooth_mean = gaussian_filter1d(mean_X, sigma=5)
    x_ticks = np.r_[1, np.arange(10, 101, 10)] # makes [1, array([10, 20, ...])] where r_ turns it into [1, 10, 20, ...]
    
    plt.xlabel("t")
    plt.xlim(1,100)
    plt.xticks(x_ticks)
    
    plt.ylim(0,mean_X.max()+1) # range of y ticks. If we want to have the same range for all plots, change to static values or the highest value in entire dataset
    plt.ylabel(r"$\bar{{X}}_t$")
    plt.gca().yaxis.set_minor_locator(MultipleLocator(1))
    
    plt.grid(which='both',alpha=0.3, zorder=1)
    plt.plot(t_vals, mean_X, marker='o', color='navy')
    plt.plot(t_vals, smooth_mean, color='orangered', zorder=1)
    plt.title("Mean spikes count over time")
    plt.show()
    
mean_plotter(X)

## Logistic reg (yoinked from Eske)

In [ ]:
def make_datasets(M, t_index, T, n, alpha, beta, gamma, lambda_0, lambda_1):

    x_features = np.empty((M, n), dtype=float)      # one row per experiment 
    y_labels = np.empty(M, dtype=int)              # class label C_t in {0, 1, 2}

    for i in range(M):
        # simulate a new HMM sequence
        C = simulate_c(gamma=gamma, beta=beta, T=T)
        Z = simulate_z(C=C, alpha=alpha, n=n)
        X = simulate_x(Z=Z, lam0=lambda_0, lam1=lambda_1)
        x_features[i] = X[t_index, :] # features are the spike counts of all neurons at time t_index
        y_labels[i] = C[t_index]     # label is the hidden state at time t_index
    
    return x_features, y_labels

### fit multiclass regression

In [ ]:
# first we generate the some dataset
x_features_data = []
y_labels_data = []
t_indexes = [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]
for t_idx in t_indexes:
    # with seed = 123 + t_idx, we get a different dataset for each t_index
    x_features, y_labels = make_datasets(M=5000, t_index=t_idx, T=big_T, n=n, alpha=alpha_val, beta=beta_val, gamma=small_gamma_val, lambda_0=lambda_0, lambda_1=lambda_1)
    x_features_data.append(x_features)
    y_labels_data.append(y_labels)
print(x_features_data[0].shape) # feature shape: (M, n)
print(y_labels_data[0].shape)   # labels shape: (M,)
print("class counts:", np.bincount(y_labels_data[0], minlength=3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

x_train_data = []
x_test_data = []
y_train_data = []
y_test_data = []

for i in range(len(t_indexes)):
    x_train, x_test, y_train, y_test = train_test_split(x_features_data[i], y_labels_data[i], test_size=0.2, random_state=123)
    x_train_data.append(x_train)
    x_test_data.append(x_test)
    y_train_data.append(y_train)
    y_test_data.append(y_test)

models = []
train_errors = []
test_errors = []

for i in range(len(t_indexes)):
    model = LogisticRegression(max_iter=1000, random_state=123)
    model.fit(x_train_data[i], y_train_data[i])
    
    train_errors.append(1 - model.score(x_train_data[i], y_train_data[i]))
    test_errors.append(1 - model.score(x_test_data[i], y_test_data[i]))
    models.append(model)

for i in range(len(t_indexes)):
    print(f"t={t_indexes[i]+1}: train_error={train_errors[i]:.6f}, test_error={test_errors[i]:.6f}")

# Part 2

To compute $C_0, C_1, \ldots, C_t$ and $Z_{0,0}, \ldots, Z_{0, i}, \ldots, Z_{t, 0}, \ldots, Z_{t,i}$ from $\mathbf{X}$, we utilize [the forward algorithm](https://en.wikipedia.org/wiki/Forward_algorithm) and the [forward/backward algorithm](https://en.wikipedia.org/wiki/Forward%E2%80%93backward_algorithm).
We first compute $P(C_t \mid \mathbf{X})$ as that will allow us to substitute parts of our $P(Z_{t,i} \mid \mathbf{X})$ when we apply Bayes.


$P(C_t = c \mid X)$ can be computed as
$$
P(C_t = c \mid X) = \frac{P(X \mid C_t = c) \cdot P(C_t = c)}{P(X)}
$$

As mentioned in [the forward algorithm](https://en.wikipedia.org/wiki/Forward_algorithm), we need to compute both our transition and emission matrix. The transition matrix is denoted as $\Gamma$ in the project, and is already given to us, however the emission matrix is not, and will need to be computed.

Our distribution for $C_t$, consists of 2 Poisson distributions; 1 for when a stimuli is "noticing something", and 1 for the complement occation. Think of it as the monkey looking with its right eye and seeing a banana, and at the same time looking with the left eye and not seeing a banana. In the same way that when you focus your eyes on something and suddenly start noticing shapes and patterns, the monkey's eyes will have more activity or not depending on this, which is where the two Poisson distributions come in. The distibutions tell us about how likely it is that the given neuron is processing a stimuli that is observing something, which is governed by the mean of the Poisson distribution ($\lambda_i$). Here we have $\lambda_0$ and $\lambda_1$, which builds up the joint distributions of the two.

Our model's general structure is
$$
C_t \to Z_{t,i} \to X_{t,i}
$$
Where we need to "eliminate" (or marginalize out) $Z$ in order to get $P(C \mid X)$, so we instead sum over all occurences for $Z$
$$
P(X_{t,i} \mid C_t) = \sum_{z\in\{0,1\}} P(Z \mid C) \cdot P(X \mid Z)
$$
This yields:
$$
\begin{aligned}
P(X_{t,i} \mid C_t) = \\ & P(Z_{t,i} = 0 \mid C_t = c) \cdot P(X_{t,i} = x \mid Z_{t,i} = 0) \\+ & P(Z_{t,i} = 1 \mid C_t = c) \cdot P(X_{t,i} = x \mid Z_{t,i} = 1)
\end{aligned}
$$
What we are computing here, can be explained as finding the probability that we are focusing with the specific stimuli in the given time step ($C_t$), and are actually observing something ($Z_{t,i} = 1$), plus the case that we aren't focusing, but still pick up on something (complement of the previous, $Z_{t,i} = 0$). Imagine it as focusing with the left eye, and observing something (giving us a large count of nerual activity spikes) and also observing something with the other eye, which isn't focused. Summing out $Z_{t,i}$ here means we get all occurences for which we actually see activity.

The cases for which we are observing something, $P(Z_{t,i} = 1 \mid C_t)$, can be described with the piecwise function as in the project:
$$
P(Z_{t,i} = 1 \mid C_t = c) = \begin{cases}
1 - \alpha & \text{ if } c = 0 \\
\alpha & \text{ if } c = 1 \\
0.5 & \text{ if } c = 2 
\end{cases}
$$
Which we can denote as a vector:
$$
[(1 - \alpha),\; \alpha,\; 0.5]
$$
Or write as:
$$
P(Z_{t,i} = 1 \mid C_t = c) = \pi_c
$$
For the complement scenario, ($z=0$), we take
$$
P(Z_{t,i} = 0 \mid C_t = c) = \begin{cases}
1-(1 - \alpha) & \text{ if } c = 0 \\
1-(\alpha) & \text{ if } c = 1 \\
1-(0.5) & \text{ if } c = 2 
\end{cases}
$$
Or simply
$$
[(1-(1 - \alpha)),\; (1-(\alpha)),\; (1-(0.5))]
$$
Which is the same as 
$$
1 - \pi_c
$$

Substituting this into our equation from earlier, we get:
$$
P(X_{t,i} \mid C_t) = (1 - \pi_c) \cdot P(X_{t,i} = x \mid Z_{t,i} = 0) + \pi_c \cdot P(X_{t,i} = x \mid Z_{t,i} = 1)
$$

If we then look at the last terms, we know (from the project task), that the distribution explaining the likelihood of the amount of spikes observed, given whether or not they came from the "active" eye or not (value of $Z_{t,i}$), is poisson distributed, with a corresponding mean to follow. Here we have 2 different distirbutions, one for when $Z_{t,i} = 0$ and one for when $Z_{t,i} = 1$. This means that the amount of spikes we see, has a likelihood which is modelled by a poisson distribution, leaving one distribution for the stimuli we are focusing on, and one for the stimuli we aren't focusing on (think of this as we might be primarily using the left eye to see a banana, there will still be a small amount of neurons processing the feedback from the right eye, and here we see the likelihood of getting the amount of spikes, given if they are from the "correct" stimuli).

For the two different distributions, we of course have 2 different lambda values (means), and so we can write the general case as:
$$
P(X_{t,i} = x \mid Z_{t,i} = z) = e^{-\lambda_z} \frac{\lambda_z^x}{x!}
$$
Which we can also just name $Pois(X_{t,i}; \lambda_z)$, which just states the likelihood of the recorded amount of spikes as mentioned before.

Combining everything we get:
$$
P(X_{t,i} = x \mid C_t = c) = (1 - \pi_c) \cdot Pois(X_{t,i}; \lambda_z) + \pi_c \cdot Pois(X_{t,i}; \lambda_z)
$$
Which again explains the likelihood of observing the given amount of spikes and processing the "correct" stimuli, plus the likelihood of observing the given amount of spikes by processing the "incorrect" stimuli, thus giving us the likelihood of observing the amount of spikes, given the stimuli chosen (state of $C_t$)

We now have the most important part of the original equation we wish to find, which is the $\textcolor{cyan}{\text{highlighted part}}$:
$$
P(C_t = c \mid \mathbf{X}) = \frac{\textcolor{cyan}{P(\mathbf{X} \mid C_t = c)} \cdot P(C_t = c)}{P(\mathbf{X})}
$$
$P(C_t = C)$ is given to us, as we know that the intial state, $t=1$ is always state $2$;
$$
P(C_1 = 2) = 1
$$
We now only have one problem:

As you might notice, the above equation has $\mathbf{X}$ instead of $X_{t,i}$, which we have. This means we need to have a single $X$ per time
 step, instead of $n$ collections. Instead of having one global $C$ state (per time step $t$), for $n$ neural activity spike observations, we instead have the same $C$ state for a unified collective <!--empire. I have brought peace,freedom, justice and security to my new empire. Your new empire?--> acitvity measurement (still likelihood), showcasing how uniform the likelihood were. If we have a lot of fairly likely measurements (given our state), but a few very unlikely cases, the very unlikely ones will "pull all the other scenarios down", in a sense that the general likelihood will be much lower.

We compute this by combining our $i$ likelihoods into a single:
$$
P(\mathbf{X_t}) = \prod_{i=0}^n P(X_{t,i})
$$
It might be easier to think of $P(X_{t,i})$ as $T \times n$ sized matrix, ($T$ rows, with $n$ columns) corresponding to $T$ time steps with $n$ observations per time step. Instead we have now collapsed it (sort of like using the collapse methods in e.g. pandas or numpy) to have a $T \times 1$ sized matrix (vector).

This part that we have now computed, that is
$$
P(X_t \mid C_t)
$$
is our emission matrix, which we can now make:

In [ ]:
from scipy.stats import poisson
def emission_matrix(X, alpha, lambda0, lambda1):
    T,n = X.shape
    pi_c = np.array([1 - alpha, alpha, 0.5])
    
    pois0 = poisson.pmf(k=X, mu=lambda0) # how likely is it that we got X, given our mean lambda 0 / 1
    pois1 = poisson.pmf(k=X, mu=lambda1) # since X is multi dimensional (entire dataset) the pois0 and pois1 will also have multiple dimensions (elementwise application)
    
    B = np.zeros(shape=(T,3), dtype=float)
    for c in range(3):
        p_i = pi_c[c]
        combined = (1 - p_i) * pois0 + (p_i) * pois1 # combined is not a single value here, but still has shape following X as before
        B[:,c] = np.prod(combined, axis=1) # all time steps with the given C state (entire column) is 
        # essentially, the np.prod() here combines all the elementwise emission probabilities (combined_{t,i}) into a single probability, as explained before in markdown cell
    return B
    

To get $X_t$ to be $\mathbf{X}$ instead, we need to use message passing, to carry all values to the "final" $X$. We do this in both the forward and backward part of the forward algorithm, which builds upon the last (or future depending on whether it's forward or backwards) entry to carry over data, thus resulting in getting $P(C_t \mid \mathbf{X})$ where $\mathbf{X}$ is an $X$ that builds upon all previous (or future) $X$'s.

### Forward / Backward Algorithm

In [ ]:
def forward(init_pi, B, Gamma):
    T = B.shape[0]
    alpha = np.zeros(shape=(T,3), dtype=float)
    scale = np.zeros(shape=T, dtype=float)
    alpha[0] = init_pi*B[0] # initial pi state: P(C_1 = c), which here would be [0, 0, 1], as P(C_1 = 2) = 1 and it must sum to 1
    # we would have alpha[0] = [0, 0, (B[0][2]] before our normalization since initial state is [0, 0, 1] leaving only the last entry in B in elementwise multiplication (hadamard product)
    scale[0] = alpha[0].sum() # we now get a scaler to normalize with
    alpha[0] = alpha[0]/scale[0] # and normalize
    
    for t in range(1, T):
        alpha[t] = B[t] * (alpha[t-1] @ Gamma) # @ is elementwise (hadamard product)
        scale[t] = alpha[t].sum()
        alpha[t] = alpha[t]/scale[t]
    
    return alpha, scale

In [ ]:
def backward(B, scale, Gamma):
    T = B.shape[0]
    beta = np.zeros(shape=(T,3), dtype=float)
    beta[-1] = 1.0
    
    for t in range(T-2, -1, -1):
        beta[t]  = B[t+1] * (beta[t+1] @ Gamma)
        beta[t]  = beta[t]/scale[t+1]
        
    return beta
    

In [ ]:
def smoothed(alpha, beta):
    if alpha.shape != beta.shape:
        raise ValueError(f"Shape mismatch, got: alpha: {alpha.shape}, beta: {beta.shape}")
    T = alpha.shape[0]
    normalizer = alpha[-1].sum()
    smoothed = (alpha * beta) / normalizer
    
    return smoothed

The `smoothed` matrix we get here, is then our
$$
P(C_t \mid \mathbf{X})
$$
Which describes the probability of having a given state at a given time step, given the observed final value which builds upon the previous observations.

Since we now know $C_t$, we can use our general HMM structure to "just emit" to get $Z_{t,i}$

To do that, we must compute the probability of getting $Z_{t,i} = 1$ given the 3 different $C_t$ states, and then multiply that with the probability of actually having those states. This utilizes message passing to first go from $\mathbf{X}$ to $C_t$, and then from $C_t$ to $Z_{t,i}$

We have
$$
P(Z_{t,i} = 1 \mid \mathbf{X}) = \sum_{c \in \{0, 1, 2\}} \left[P(Z_{t,i} \mid X_{t,i}, C_t = c) \cdot P(C_t \mid \mathbf{X})\right]
$$
But if $C_t$ is given, $Z_{t,i} \perp X_{t,i}$, so we have:
$$
P(Z_{t,i} = 1 \mid \mathbf{X}) = P(Z_{t,i} = 1 \mid C_t = c) \cdot P(C_t \mid \mathbf{X}) 
$$
Where we just computed $P(C_t \mid \mathbf{X})$ (our smoothed), and the

In [ ]:
def posterior_z(X, alpha, lambda0, lambda1, smoothed):
    T, n = X.shape
    pi_c = np.array([1 - alpha, alpha, 0.5])
    
    pois0 = poisson.pmf(k=X, mu=lambda0) # how likely is it that we got X, given our mean lambda 0 / 1
    pois1 = poisson.pmf(k=X, mu=lambda1) # since X is multi dimensional (entire dataset) the pois0 and pois1 will also have multiple dimensions (elementwise application)
    
    z1_given_c = np.zeros(shape=(T,n,3)) # T time steps, n neurons and 3 states for each
    
    for c in range(3):
        p_i = pi_c[c]
        denominator = (1 - p_i) * pois0 + p_i * pois1 # probability of observing X spikes given value of Z * probability of having Z value given state C for the case where we process the correct stimuli and the case where we don't
        z1_given_c[:, :, c] = (p_i * pois1) / denominator # the case where we process correct stimuli divided by both cases, so we normalize it
        
    z1 = np.zeros(shape=(T,n), dtype=float)
    
    for t in range(T):
        for i in range(n):
            z1[t, i] = (z1_given_c[t, i, 0] * smoothed[t, 0] + z1_given_c[t, i, 1] * smoothed[t, 1] + z1_given_c[t, i, 2] * smoothed[t, 2])
            # the probability of getting z = 1 given c = 0 + z = 1 given c = 1 + z = 1 given c = 2
            # which is the probability of getting z = 1 given all the different states, for the single z_{t,i}
    
    return z1

In [ ]:
def pipeline(X, alpha, beta, gamma, lambda0, lambda1, T, n):
    return